# Import

Import all necessary libraries. If you face any issue, please install the libraries using pip or conda.

Ensure reproducibility of your code and keep reusing your functions in all assignments.

**Note: NetworkX must not be used in your solution, and it is only for checking your results.**

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import networkx as nx


In [4]:
import sys
print(sys.executable)


/Users/ronahnakonde/Documents/Analysis_of_Complex_Networks/Week_2/venv/bin/python


In [6]:
# change this to the filename you want to read
undirected_network_edges = "undirected_network_edge_list.csv"
directed_network_edges = "directed_network_edge_list.csv"

# don't change this part
if len(sys.argv) > 2 and 'ipykernel' not in sys.modules:
    print("Running in shell, reading filename from command line argument")
    undirected_network_edges = sys.argv[1]
    directed_network_edges = sys.argv[2]

In [7]:
def read_network(filename, directed=False):
    """
    Read the edge list from filename into adjacency matrix, node list, and NetworkX graph.
    If directed is True, treat the network as directed.
    Args:
        filename: path to the edge list file (CSV format)
        directed: boolean indicating if the network is directed
    Returns:
        adj: adjacency matrix (numpy array)
        node_list: sorted unique node IDs (numpy array)
        G: NetworkX graph
    """

    pass # paste your function in previous assignment here
    # read edge list into df
    df = pd.read_csv(undirected_network_edges)
    
    # sort the node IDs into node_list
    all_nodes = set(df['from']) | set(df['to'])
    node_list = np.array(sorted(list(all_nodes)))
    
    # convert df into adjacency matrix adj ordered by node_list
    n = len(node_list)
    adj = np.zeros((n, n), dtype=int)
    
    node_to_index = {node: i for i, node in enumerate(node_list)}
    
    # Fill the adjacency matrix
    for _, row in df.iterrows():
        source = row['from']
        target = row['to']
        i = node_to_index[source]
        j = node_to_index[target]
        adj[i, j] = 1
        
        # If undirected, also set the symmetric entry
        if not directed:
            adj[j, i] = 1
    
    # read the edge list using NetworkX into G
    if directed:
        G = nx.from_pandas_edgelist(df, source='from', target='to', edge_attr='weight', 
                                   create_using=nx.DiGraph())
    else:
        G = nx.from_pandas_edgelist(df, source='from', target='to', edge_attr='weight')
    
    return adj, node_list, G
   

In [8]:
adj, node_list, G = read_network(undirected_network_edges, directed=False)

# Question 1 (4 marks)

Implement a function to compute the clustering coefficient of each node and the average clustering coefficient from the adjacency matrix.

In [9]:
def clustering_coefficient(adj, is_directed=False):
    """
    Compute the (average) clustering coefficient of each node from the adjacency matrix.
    Hint: Symmetrize the adjacency matrix first if the network is directed.
    Args:
        adj: adjacency matrix (numpy array)
        is_directed: boolean indicating if the network is directed
    Returns:
        clustering_coeffs: array of clustering coefficients (numpy array)
        average_clustering_coeff: average clustering coefficient (float)
    """
    pass
    # Step 1: Symmetrize if directed (treat as undirected for clustering)
    if is_directed:
        adj_sym = (adj + adj.T > 0).astype(int)  # Union of edges in both directions
    else:
        adj_sym = adj
    
    n = adj_sym.shape[0]  # number of nodes
    clustering_coeffs = np.zeros(n)
    
    # Step 2: For each node, compute its clustering coefficient
    for i in range(n):
        # Find neighbors of node i
        neighbors = np.where(adj_sym[i] == 1)[0]
        k_i = len(neighbors)  # degree of node i
        
        # If node has 0 or 1 neighbors, clustering coefficient is 0 (undefined, set to 0)
        if k_i < 2:
            clustering_coeffs[i] = 0.0
            continue
        
        # Step 3: Count edges between neighbors
        # Extract the subgraph of neighbors
        edges_between_neighbors = 0
        for idx1 in range(len(neighbors)):
            for idx2 in range(idx1 + 1, len(neighbors)):
                neighbor1 = neighbors[idx1]
                neighbor2 = neighbors[idx2]
                # Check if there's an edge between neighbor1 and neighbor2
                if adj_sym[neighbor1, neighbor2] == 1:
                    edges_between_neighbors += 1
        
        # Step 4: Compute clustering coefficient
        # Maximum possible edges = k_i * (k_i - 1) / 2
        max_possible_edges = k_i * (k_i - 1) / 2
        clustering_coeffs[i] = edges_between_neighbors / max_possible_edges
    
    # Step 5: Compute average clustering coefficient
    average_clustering_coeff = np.mean(clustering_coeffs)
    
    return clustering_coeffs, average_clustering_coeff

In [10]:
assert np.allclose(clustering_coefficient(adj)[0],
                   pd.Series(dict(nx.clustering(G))).sort_index().values), "Question 1: Your function is not correct!"
print("Question 1: Your function is correct!")

Question 1: Your function is correct!


In [11]:
print("Average clustering coefficient:", clustering_coefficient(adj)[1])

Average clustering coefficient: 0.4960095935883329


# Question 2 (4 marks)

Implement a function to find the largest connected component of the graph from the adjacency matrix.

In [14]:
def largest_connected_component(adj):
    """
    Extract the largest connected component from the adjacency matrix.
    Hint: Node IDs may not be continuous, use node_list to map the adj indices back to original node IDs.
    Args:
        adj: adjacency matrix (numpy array)
    Returns:
        lcc_node_list: node set of the largest connected component (set)
    """
    pass
    n = adj.shape[0]  # number of nodes
    visited = np.zeros(n, dtype=bool)  # track visited nodes
    components = []  # store all connected components
    
    # Helper function: BFS to find one connected component starting from node 'start'
    def bfs(start):
        component = set()
        queue = [start]
        visited[start] = True
        
        while queue:
            node = queue.pop(0)
            component.add(node)
            
            # Find all neighbors of current node
            neighbors = np.where(adj[node] == 1)[0]
            
            for neighbor in neighbors:
                if not visited[neighbor]:
                    visited[neighbor] = True
                    queue.append(neighbor)
        
        return component
    
    # Find all connected components
    for node in range(n):
        if not visited[node]:
            component = bfs(node)
            components.append(component)
    
    # Find the largest connected component (in terms of indices)
    lcc_indices = max(components, key=len)
    
    # Map indices back to original node IDs using node_list
    lcc_node_list = {node_list[idx] for idx in lcc_indices}
    
    return lcc_node_list

In [15]:
assert(max(nx.connected_components(G), key=len) == largest_connected_component(adj)), "Question 2: Your function is not correct!"
print("Question 2: Your function is correct!")

Question 2: Your function is correct!


# Question 3 (4 marks)

Implement a function to generate an undirected Erdős-Rényi (ER) random graph `G(n,p)` given the number of nodes `n` and the probability of edge `p`.

The function should return the adjacency matrix of the generated graph.

1) Fix the random seed
2) Generate the graph
3) Remove self-loops
4) Make the graph undirected (Hint: use `np.triu` or `np.tril` to get the upper or lower triangular part of `adj`)
5) Return the adjacency matrix `adj`

In [16]:
def ER_graph_generator(n, p, seed=0):
    """
    Generate an undirected Erdős-Rényi (ER) random graph with n nodes and edge probability p.
    Args:
        n: number of nodes (int)
        p: probability of edge creation (float)
    Returns:
        adj: adjacency matrix of the generated ER graph (numpy array)
    """
    rng = np.random.Generator(np.random.PCG64(seed))
    pass
    # Step 1: Fix the random seed
    rng = np.random.Generator(np.random.PCG64(seed))
    
    # Step 2: Generate the graph
    # Create random matrix where each entry is 1 with probability p
    adj = (rng.random((n, n)) < p).astype(int)
    
    # Step 3: Remove self-loops (set diagonal to 0)
    np.fill_diagonal(adj, 0)
    
    # Step 4: Make the graph undirected
    # Use upper triangular part and mirror it to lower triangular part
    adj = np.triu(adj)  # Keep only upper triangular part
    adj = adj + adj.T   # Add transpose to make symmetric
    
    # Step 5: Return the adjacency matrix
    return adj

# Question 4 (2 marks)

1) Simulate 50 ER graphs with `n=100` and `p=0.3`, and compute the average `s` and standard deviation `std` of the clustering coefficient for each graph using your function from Question 2.
2) Compare your results with the theoretical expectation of the clustering coefficient of ER graphs, which is equal to `p`.
3) Compute the z-score `(s-p)/std` of your results.

Your z-score should be cloose to 0, and it has a 95% probability of being in the range `[-1.96, 1.96]`.

In [17]:
n = 100
p = 0.3
clustering_coeff = []

# Simulate 50 ER graphs
for seed in range(50):
    # Generate ER graph
    adj = ER_graph_generator(n, p, seed=seed)
    
    # Compute average clustering coefficient
    _, avg_clustering = clustering_coefficient(adj, is_directed=False)
    
    # Store the result
    clustering_coeff.append(avg_clustering)

# Convert to numpy array for easy calculations
clustering_coeff = np.array(clustering_coeff)

# Compute statistics
s = np.mean(clustering_coeff)
std = np.std(clustering_coeff, ddof=1)  # Use ddof=1 for sample standard deviation

# Compute z-score
z_score = (s - p) / std

print("Mean clustering coefficient from simulations:", s)
print("Standard deviation of clustering coefficient from simulations:", std)
print("Theoretical clustering coefficient (p):", p)
print("Z-score:", z_score)

Mean clustering coefficient from simulations: 0.3005775105937504
Standard deviation of clustering coefficient from simulations: 0.006857801466064241
Theoretical clustering coefficient (p): 0.3
Z-score: 0.08421220658081029


# Question 5 (2 marks)

Design functions to extract the degree sequence from an adjacency matrix.

In [23]:
def degree(adj):
    """
    Compute the degree of each node from the adjacency matrix.
    Args:
        adj: adjacency matrix (numpy array)
    Returns:
        degrees: array of node degrees (numpy array)
    """
    pass
    degrees = np.sum(adj, axis=1)
    return degrees

In [25]:
# check your function using NetworkX
assert np.allclose(degree(adj),
                   pd.DataFrame(G.degree()).sort_values(by=0).values[:,1]), "Question 5: Your function is not correct!"
print("Question 5: Your function is correct!")

Question 5: Your function is correct!


# Question 6 (4 marks)

Design a function to generate a undirected random graph using the configuration model given a degree sequence.
Your graph may contain self-loops and multi-edges.

1) Fix the random seed
2) Initialize the graph with no edges
3) Adding edges between nodes randomly while respecting the degree sequence
4) Update the degree sequence after adding each edge
5) Return the adjacency matrix of the generated graph
6) Try to generate a random graph with the degree sequence of `adj`

In [26]:
def Configuration_model_generator(deg_seq, seed=0):
    """
    Generate an undirected random graph using the configuration model with a given degree sequence.
    Your graph may contain self-loops and multi-edges.
    Args:
        deg_seq: degree sequence (list or numpy array)
        seed: random seed (int)
    Returns:
        adj: adjacency matrix of the generated graph (numpy array)
    """
    pass
    # Step 1: Fix the random seed
    rng = np.random.Generator(np.random.PCG64(seed))
    
    # Step 2: Initialize the graph with no edges
    n = len(deg_seq)
    adj = np.zeros((n, n), dtype=int)
    
    # Step 3: Create a list of stubs (half-edges)
    # Each node i appears deg_seq[i] times in the stub list
    stubs = []
    for node, degree in enumerate(deg_seq):
        stubs.extend([node] * degree)
    
    # Convert to numpy array for easier manipulation
    stubs = np.array(stubs)
    
    # Step 4: Randomly shuffle the stubs
    rng.shuffle(stubs)
    
    # Step 5: Pair up stubs to create edges
    # Take pairs of stubs and connect them
    for i in range(0, len(stubs) - 1, 2):
        node1 = stubs[i]
        node2 = stubs[i + 1]
        
        # Add edge (allows self-loops and multi-edges)
        adj[node1, node2] += 1
        adj[node2, node1] += 1
    
    # Step 6: Return the adjacency matrix
    return adj

In [27]:
# check your function using NetworkX

cm_graph_adj = Configuration_model_generator(degree(adj), seed=0)
assert np.allclose(degree(cm_graph_adj),
                   pd.DataFrame(G.degree()).sort_values(by=0).values[:,1]), "Question 6: Your function is not correct!"
print("Question 6: Your function is correct!")

Question 6: Your function is correct!


# Visualization

Run this cell to visualize the graph.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 10))
axes = axes.flatten()

er_graph_adj = ER_graph_generator(n=adj.shape[0], p=adj.mean(), seed=0)

# visualize the degree distribution
axes[0].scatter(range(adj.shape[0]), sorted(degree(adj), reverse=True), s=2)
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_title('Degree Distribution of Original Graph')

axes[1].scatter(range(er_graph_adj.shape[0]), sorted(degree(er_graph_adj), reverse=True), s=2)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title('Degree Distribution of ER Graph')

axes[2].scatter(range(cm_graph_adj.shape[0]), sorted(degree(cm_graph_adj), reverse=True), s=2)
axes[2].set_xscale('log')
axes[2].set_yscale('log')
axes[2].set_title('Degree Distribution of CM Graph')

nx.draw_networkx(nx.from_numpy_array(adj), ax=axes[3], node_size=10, with_labels=False)
axes[3].set_title('Original Graph Visualization')

nx.draw_networkx(nx.from_numpy_array(er_graph_adj), ax=axes[4], node_size=10, with_labels=False)
axes[4].set_title('ER Graph Visualization')

nx.draw_networkx(nx.from_numpy_array(cm_graph_adj), ax=axes[5], node_size=10, with_labels=False)
axes[5].set_title('CM Graph Visualization')

fig.tight_layout()